# Data Collection & Web Scraping

Scrapes the raw data for this project into `data/`:

- **Goodreads**: books and reviews (`goodreads-books.csv`, `goodreads-reviews.csv`)
- **Rotten Tomatoes**: movies, critic reviews and audience reviews (`rotten-tomatoes-*.csv`)

The cleaned versions are produced by `data-cleaning.ipynb`, and the analysis is in `main.ipynb`.

### Contents
1. **Infrastructure:** HTTP clients with retry and backoff, logging, cache helpers and a streaming `CsvWriter`.
2. **Goodreads pipeline:** Listopia list discovery, then async scraping of book pages and reviews.
3. **Rotten Tomatoes pipeline:** Playwright for the JavaScript browse pages, then async scraping of movie and review pages.
4. **Pipeline registry:** run one or both scrapers with `run_pipelines([...])`.

Run locally (Playwright needs a real browser): `pip install -r requirements.txt && playwright install chromium`.

# Introduction

In this cell, we have made a scraping framework that combines two independent scrapers into one
unified codebase with shared infrastructure. This design makes it very easy to add new
sources (IMDb, Amazon, etc.) by subclassing a few abstract base classes, that way, in case we add more external sources or remove some, it will not disrupt the pipeline

## Imports & Dependencies

In [ ]:
from __future__ import annotations  # lets us use str | None syntax on older Pythons

import asyncio  # async event loop for concurrent HTTP requests
import csv      # CSV file writing
import json     # parsing JSON payloads embedded in HTML pages
import logging  # structured log output to console and file
import os       # filesystem operations (makedirs, path checks)
import random   # randomize user-agent selection + jitter delays
import re       # regex for extracting IDs and patterns from HTML
import time     # monotonic clock for rate-limit tracking, sleep for sync client
# import sys      # CLI argument parsing for selecting which scraper to run
from abc import ABC, abstractmethod   # abstract base classes for the plugin pattern
from dataclasses import asdict, dataclass, fields  # dataclasses → CSV rows via asdict()
from typing import Any, TypeVar  # generic type variable for cache loader

import aiohttp     # async HTTP client — non-blocking, supports concurrent requests
import requests    # sync HTTP client — simpler, used for small discovery phases
import bs4      # BeautifulSoup — HTML parsing (used with lxml backend for speed)

# Playwright is only needed for Rotten Tomatoes (JS-rendered pages).
# We import it lazily inside the RT module so the Goodreads scraper
# works even if Playwright isn't installed.
try:
    from playwright.sync_api import sync_playwright, Page
    PLAYWRIGHT_AVAILABLE = True
except ImportError:
    PLAYWRIGHT_AVAILABLE = False
    print("⚠ Playwright not installed — Rotten Tomatoes scraper will not work.")
    print("  Install it with: pip install playwright && playwright install chromium")
# --- Notebooks compatibility ---
# notebooks already run inside an asyncio event loop, so calling
# asyncio.run() would fail with "cannot be called from a running event loop".
# nest_asyncio patches the loop to allow nested runs, making the same code
# work in both .py scripts (terminal) and .ipynb notebooks (Jupyter)
# so async operations can now accurately work.
import nest_asyncio
nest_asyncio.apply()

DATA_DIR = "data"  # raw CSVs and caches are written here
os.makedirs(DATA_DIR, exist_ok=True)


In [ ]:
os.listdir(DATA_DIR)

## Logging Setup
- every scraper module gets its own logger which makes it easier to know what is happening and spot any issues quickly. the output shows INFO+ while an optional log file captures DEBUG-level details incase anything goes wrong

In [ ]:
def setup_logger(name: str, log_file: str | None = None) -> logging.Logger:
    lgr = logging.getLogger(name)
    lgr.setLevel(logging.DEBUG)  # capture everything; handlers filter what they show
    lgr.propagate = False
    # only add handlers once — avoids duplicate log lines if the cell is re-run
    if not lgr.handlers:
        console = logging.StreamHandler()
        console.setLevel(logging.INFO)
        console.setFormatter(logging.Formatter(
            "%(asctime)s [%(name)s:%(levelname)s] %(message)s", datefmt="%H:%M:%S"
        ))
        lgr.addHandler(console)

        if log_file:
            fh = logging.FileHandler(log_file, mode="w", encoding="utf-8")
            fh.setLevel(logging.DEBUG)
            fh.setFormatter(logging.Formatter(
                "%(asctime)s [%(name)s:%(levelname)s] %(message)s",
                datefmt="%Y-%m-%d %H:%M:%S",
            ))
            lgr.addHandler(fh)

    return lgr

# create the default logger for the notebook
logger = setup_logger("MAIN-LOGGER")

In [ ]:
# test out the logegr to ensure it works
logger.info("This is an info message")

## Base Configurations & HTTP Clients
**Base Configuration**:

- we create a class to hold setting configs every scraper would need including rate limiting, retry params, output directories, specific parameters a scraper may need and a group of user agents. We make the configuration immutable after creation so nothing is changed

**HTTP CLIENTS**:

- now, you can ideally make a simple request using request.<operation> to a webpage but there are many factors that could stop us from scraping a webstie especially since we need a lot of data. In addition, since we are doing this asynchronous so we aren't waiting a few hours until the scraping is done, we need to set up some guardrails so our ip is not blocked from accessing the site. In this next few notebooks we have:
  - **SyncHttpClient**: This is used for small phases where the data won't take long to collect. It processes requests one at a time and blocks the execution thread meaning it waits for a complete response from the server before initiating next requests. We use this client for lightweight tasks.
  - **AsyncHttpClient**: This handles multiple network requests simultaneously such that scraping is faster. It uses semaphore caps which implements strict concurrency limits i.e. the max number of requests to prevent overloading the target server. It uses expoential backoff and retry0after support which manages failed requests by dynamically increasing wait time between retries. It utilizes a shared state across the concurrent poll such that if a singel request triggers a 239 too many requests response, all concurrent operations pause preventing ip bans. then just incase, we have cloudflare handling so we handle access denials and bot protections from cloudfare

In [ ]:
@dataclass(frozen=True)
class BaseScraperConfig:
    """Common configuration fields shared across all scraper sources."""

    # --- rate limiting ---
    request_delay: float = 1.5    # seconds to wait between requests (prevents bans)
    max_retries: int = 4         # how many times to retry a failed request
    retry_backoff: float = 3.0     # exponential backoff base: 3s, 9s, 27s, 81s
    default_rate_limit_wait: float = 60.0  # fallback wait when 429 has no Retry-After header

    # --- concurrency ---
    concurrency: int = 6   # max simultaneous async requests (6 at a time)

    # --- output ---
    output_dir: str = DATA_DIR   # where CSV files and caches are written
    flush_every: int = 25     # write buffered rows to disk every N items

    # --- user agents ---
    # we rotate these per-request so the target site sees different "browsers"
    # and is less likely to fingerprint and block us
    user_agents: tuple[str, ...] = (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4 Safari/605.1.15",
    )


In [ ]:
def browser_headers(user_agent: str) -> dict[str, str]:
    """Build a full set of HTTP headers that mimic a real browser.

    Why bother? Sites like Goodreads and RT use Cloudflare or similar
    that check for missing Accept-Language, Sec-Fetch-* headers,
    etc. Sending realistic headers reduces blocking.
    """
    return {
        "User-Agent": user_agent, # identifies the "browser" type
        "Accept": (     # tells server what content types we want
            "text/html,application/xhtml+xml,application/xml;"
            "q=0.9,image/webp,*/*;q=0.8"
        ),
        "Accept-Language": "en-US,en;q=0.9",    # English language preference
        "Accept-Encoding": "gzip, deflate, br", # we can handle compressed responses
        "Cache-Control": "no-cache",  # don't serve us stale cached pages
        "Pragma": "no-cache",   # same as above, for HTTP/1.0 compat
        "Sec-Fetch-Dest": "document",  # browser navigation metadata
        "Sec-Fetch-Mode": "navigate",  # we're navigating to a page
        "Sec-Fetch-Site": "none",     # not coming from another site
        "Sec-Fetch-User": "?1",     # user-initiated request
        "Upgrade-Insecure-Requests": "1", # prefer HTTPS
    }

In [ ]:
class SyncHttpClient:
    """Blocking HTTP client with retry + exponential backoff.

    Used for small-volume discovery phases where async is overkill.
    like Goodreads Phase 1a fetches ~90 genre index pages sequentially.
    """

    def __init__(
        self,
        user_agents: tuple[str, ...],  # pool of user-agent strings to rotate through
        delay: float = 1.5,            # minimum seconds between requests
        max_retries: int = 4,          # give up after this many failures
        retry_backoff: float = 3.0,    # exponential backoff base (3^1=3s, 3^2=9s, ...)
    ):
        self._session = requests.Session()  # reuses TCP connections for speed
        self._user_agents = user_agents
        self._delay = delay
        self._max_retries = max_retries
        self._retry_backoff = retry_backoff

    def get(self, url: str) -> str:
        """Fetch a URL, retrying on 429s and server errors."""
        for attempt in range(self._max_retries + 1):
            time.sleep(self._delay)  # polite delay before every request
            ua = random.choice(self._user_agents)  # pick a random user-agent each time
            try:
                resp = self._session.get(
                    url, headers=browser_headers(ua), timeout=30  # 30s timeout per request
                )

                # --- 429 Too Many Requests: the site is rate-limiting us ---
                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")  # server may tell us how long to wait
                    wait = (
                        float(retry_after)                         # use server's suggestion if available
                        if retry_after
                        else self._retry_backoff ** (attempt + 1)  # otherwise exponential backoff
                    )
                    logger.warning("429 (sync) on %s, waiting %.1fs", url, wait)
                    time.sleep(wait)
                    continue  # retry the request

                # --- 5xx Server Error: the server is having problems ---
                if resp.status_code >= 500:
                    wait = self._retry_backoff ** (attempt + 1)
                    logger.warning("HTTP %d on %s, retrying in %.1fs", resp.status_code, url, wait)
                    time.sleep(wait)
                    continue  # retry

                resp.raise_for_status()  # raise exception for any other non-2xx status
                return resp.text         # success — return the HTML

            except requests.RequestException as e:
                if attempt == self._max_retries:
                    raise  # give up after max retries
                wait = self._retry_backoff ** (attempt + 1)
                logger.warning(
                    "Sync request error on %s (attempt %d/%d): %s, retrying in %.1fs",
                    url, attempt + 1, self._max_retries, e, wait,
                )
                time.sleep(wait)

        raise RuntimeError(f"All retries exhausted for {url}")

In [ ]:
class AsyncHttpClient:
    """Non-blocking HTTP client with concurrency control, retry, and
    global rate-limit cool-off.

    This is the workhorse for Phase 2 (detail scraping) where we're
    hitting thousands of pages so the scraping can be fast.
    Key features:

    - Semaphore caps how many requests fly simultaneously
    - Exponential backoff on 429, 403, and 5xx errors
    - Honors Retry-After header when the server sends one
    - Global cool-off: when ONE request gets a 429, ALL concurrent
      requests pause too (prevents cascading 429s)
    - Jittered delay: adds random 0-0.5s to the base delay so
      requests don't fire in synchronized bursts
    """

    def __init__(
        self,
        user_agents: tuple[str, ...],            # pool of UAs to rotate through
        delay: float = 1.5,                      # base delay between requests
        max_retries: int = 4,                    # max retry attempts per URL
        retry_backoff: float = 3.0,              # backoff multiplier (3^1, 3^2, ...)
        default_rate_limit_wait: float = 60.0,   # fallback wait for 429 without Retry-After
    ):
        self._user_agents = user_agents
        self._delay = delay
        self._max_retries = max_retries
        self._retry_backoff = retry_backoff
        self._default_rate_limit_wait = default_rate_limit_wait
        self._session: aiohttp.ClientSession | None = None    # created in start()
        self._semaphore: asyncio.Semaphore | None = None      # created in start()
        self._rate_limit_until: float = 0.0  # monotonic timestamp: all requests pause until this time

    async def start(self, concurrency: int) -> None:
        """Initialize the aiohttp session and concurrency semaphore.

        Must be called before any get() calls. Separated from __init__
        because async operations can't happen in a constructor.
        """
        self._semaphore = asyncio.Semaphore(concurrency)  # at most N requests in flight
        connector = aiohttp.TCPConnector(limit=concurrency * 2)  # TCP connection pool (2× concurrency headroom)
        self._session = aiohttp.ClientSession(
            connector=connector,
            timeout=aiohttp.ClientTimeout(total=45),  # 45s total timeout per request
        )

    async def close(self) -> None:
        """Shut down the aiohttp session and release connections."""
        if self._session:
            await self._session.close()

    async def _wait_if_rate_limited(self) -> None:
        """If a recent 429 triggered a global cool-off, sleep until it expires.

        This prevents other concurrent tasks from hammering the server
        while it's already telling us to slow down.
        """
        now = time.monotonic()
        if now < self._rate_limit_until:
            wait = self._rate_limit_until - now  # how much longer to wait
            logger.debug("Honoring global rate-limit cool-off: %.1fs", wait)
            await asyncio.sleep(wait)

    def _set_rate_limit_cooloff(self, seconds: float) -> None:
        """Set a global cool-off: ALL requests pause for `seconds` from now.

        Only extends the deadline — never shortens it. This way if two
        429s arrive close together, we honor the longer wait.
        """
        target = time.monotonic() + seconds
        if target > self._rate_limit_until:
            self._rate_limit_until = target

    async def get(self, url: str) -> str:
        """Fetch a URL with rate limiting, retry, and global cool-off."""
        async with self._semaphore:  # block here if we're at max concurrency
            for attempt in range(self._max_retries + 1):
                await self._wait_if_rate_limited()  # honor any active global cool-off

                # jittered delay: base + random 0-0.5s prevents synchronized bursts
                await asyncio.sleep(self._delay + random.uniform(0, 0.5))

                ua = random.choice(self._user_agents)  # rotate user-agent per request
                try:
                    async with self._session.get(url, headers=browser_headers(ua)) as resp:

                        # --- 429 Too Many Requests ---
                        if resp.status == 429:
                            retry_after = resp.headers.get("Retry-After")
                            if retry_after:
                                try:
                                    wait = float(retry_after)  # server says "wait N seconds"
                                except ValueError:
                                    wait = self._default_rate_limit_wait  # header was malformed
                            else:
                                wait = self._retry_backoff ** (attempt + 1)  # exponential backoff
                            self._set_rate_limit_cooloff(wait)  # pause ALL concurrent requests too
                            logger.warning(
                                "429 on %s, cooling off %.1fs (attempt %d/%d)",
                                url, wait, attempt + 1, self._max_retries,
                            )
                            await asyncio.sleep(wait)
                            continue

                        # --- 403 Forbidden (usually Cloudflare challenge) ---
                        if resp.status == 403:
                            wait = self._default_rate_limit_wait  # back way off
                            self._set_rate_limit_cooloff(wait)    # pause everyone
                            logger.warning("403 on %s (likely Cloudflare), cooling off %.1fs", url, wait)
                            await asyncio.sleep(wait)
                            continue

                        # --- 5xx Server Error ---
                        if resp.status >= 500:
                            wait = self._retry_backoff ** (attempt + 1)
                            logger.warning("HTTP %d on %s, retrying in %.1fs", resp.status, url, wait)
                            await asyncio.sleep(wait)
                            continue

                        resp.raise_for_status()      # raise on any other non-2xx
                        return await resp.text()     # success!

                except Exception as e:
                    if attempt == self._max_retries:
                        raise  # exhausted all retries — let the caller handle it
                    wait = self._retry_backoff ** (attempt + 1)
                    logger.warning(
                        "Request error on %s (attempt %d/%d): %s: %s, retrying in %.1fs",
                        url, attempt + 1, self._max_retries, type(e).__name__, e, wait,
                    )
                    await asyncio.sleep(wait)

            raise RuntimeError(f"All retries exhausted for {url}")


## CSV writer
Streaming writer that accepts any `@dataclass` type. It inspects the dataclass
fields to build the CSV header, then converts each instance to a dict row via
`asdict()`. Flushes to disk on every `append()` call so data is available even
if the scraper crashes mid-run.

In [ ]:
class CsvWriter:
    """Streaming CSV writer for dataclass instances.

    Usage:
        writer = CsvWriter("output/books.csv", BookDetail)
        writer.open()          # creates file, writes header row
        writer.append([b1, b2])  # converts dataclasses → dicts → CSV rows, flushes
        writer.close()         # closes file, logs row count
    """

    def __init__(self, filepath: str, cls: type):
        self._filepath = filepath
        self._cls = cls
        # extract field names from the dataclass to use as CSV column headers
        self._fieldnames = [f.name for f in fields(cls)]
        self._file = None      # file handle, set in open()
        self._writer = None    # csv.DictWriter, set in open()
        self._count = 0        # running count of rows written

    def open(self) -> None:
        """Open the CSV file. If it already exists, append to it.
        If it doesn't exist, create it and write the header."""
        os.makedirs(os.path.dirname(self._filepath) or ".", exist_ok=True)

        if os.path.exists(self._filepath):
            # --- RESUME MODE: file exists from a previous run ---
            # count how many rows are already there
            with open(self._filepath, "r", encoding="utf-8") as f:
                self._count = sum(1 for _ in f) - 1  # subtract the header row
            if self._count < 0:
                self._count = 0
            # open in append mode — no header written
            self._file = open(self._filepath, "a", newline="", encoding="utf-8")
            self._writer = csv.DictWriter(self._file, fieldnames=self._fieldnames)
            logger.info("Resuming CSV: %s (%d rows already present)", self._filepath, self._count)
        else:
            # --- FRESH MODE: new file ---
            self._file = open(self._filepath, "w", newline="", encoding="utf-8")
            self._writer = csv.DictWriter(self._file, fieldnames=self._fieldnames)
            self._writer.writeheader()
            self._count = 0

    def get_existing_values(self, column: str) -> set[str]:
        """Read the existing CSV and return all values from the given column.
        Call this AFTER open() to find out what's already been scraped."""
        values: set[str] = set()
        if not os.path.exists(self._filepath):
            return values
        try:
            with open(self._filepath, "r", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    val = row.get(column, "")
                    if val:
                        values.add(val)
        except Exception as e:
            logger.warning("Could not read existing CSV for resume check: %s", e)
        return values

    def append(self, rows: list) -> None:
        """Convert dataclass instances to dicts and write them as CSV rows."""
        if not self._writer:
            raise RuntimeError("CsvWriter not opened — call open() first")
        for row in rows:
            self._writer.writerow(asdict(row))  # dataclass to dict to CSV row
            self._count += 1
        self._file.flush()  # force write to disk so data survives crashes

    def close(self) -> None:
        """Close the file and log how many rows were written."""
        if self._file:
            self._file.close()
            logger.info("Saved %d rows to %s", self._count, self._filepath)
            self._file = None
            self._writer = None


## Cache Helpers

Scraping thousands of pages takes hours. If the process crashes halfway through
Phase 2, we don't want to re-discover all the URLs again. These helpers save
discovered listings to a JSON file (cache) after Phase 1 completes. On re-run,
if the cache exists, Phase 1 is skipped entirely.

In [ ]:
T = TypeVar("T")  # generic type for the cache loader

In [ ]:
def save_cache(path: str, items: list, dedup_key: str | None = None) -> None:
    """Save a list of dataclass instances to a JSON file.

    Args:
        path:      where to write the JSON file
        items:     list of dataclass instances to persist
        dedup_key: if set, only keep the first occurrence of each value
                   for this field (e.g. dedup_key="book_id" removes duplicate books)
    """
    try:
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)  # create parent dirs if needed
        dicts: list[dict] = []
        seen: set[str] = set()  # tracks which dedup_key values we've already added
        for item in items:
            d = asdict(item)  # convert dataclass → dict for JSON serialization
            if dedup_key:
                val = d.get(dedup_key, "")
                if val and val in seen:
                    continue  # skip duplicate
                seen.add(val)
            dicts.append(d)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(dicts, f, ensure_ascii=False)  # write to disk
        logger.info("Cache saved: %d items → %s", len(dicts), path)
    except Exception as e:
        logger.warning("Failed to save cache: %s", e)

In [ ]:
def load_cache(path: str, cls: type[T]) -> list[T] | None:
    """Load cached dataclass instances from a JSON file.

    Returns None if the file doesn't exist or is corrupt — the pipeline
    will then fall through to a full discovery run.
    """
    if not os.path.exists(path):
        return None  # no cache file → run full discovery
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)  # list of dicts
        return [cls(**row) for row in data]  # reconstruct dataclass instances
    except (json.JSONDecodeError, TypeError, KeyError) as e:
        logger.warning("Cache file corrupt, ignoring: %s", e)
        return None

## Abstract Base Classes

These define the plugin contract that every scraper source should implement.
The beauty of this is that now, the framework doesn't care what you are scraping, as long as your classes implement these three interfaces, the pipeline operations (queue bridging, CSV streaming, caching) works automatically.

In [ ]:
class BaseDiscoverer(ABC):
    """Phase 1: discover item URLs to scrape.

    Each source implements this differently:
      - Goodreads: crawl genre index pages → list pages → book URLs
      - Rotten Tomatoes: scroll through JS-rendered browse pages → movie URLs
      - A hypothetical Amazon scraper: search result pages → product URLs
    """

    @abstractmethod
    def discover(self) -> list[Any]:
        """Return a list of lightweight listing objects (BookListing, MovieListing, etc.)
        that will be scraped in the detail phase."""
        ...

In [ ]:
class BaseDetailScraper(ABC):
    """Phase 2: scrape full detail pages from discovered listings.

    Consumes items from an asyncio.Queue (fed by Phase 1) and writes
    results to one or more CsvWriters.
    """

    @abstractmethod
    async def scrape_from_queue(
        self,
        queue: asyncio.Queue,   # items arrive here from the discoverer
        *writers: CsvWriter,    # one or more CSV writers for different output types
    ) -> tuple[int, ...]:
        """Consume listings from the queue, scrape each one, write to CSV.

        Returns a tuple of counts (e.g. (books_scraped, reviews_scraped)).
        A None sentinel in the queue signals "no more items coming".
        """
        ...


In [ ]:
class BasePipeline(ABC):
    """Top-level orchestrator that wires discovery → detail scraping → CSV output.

    Each source implements run() to set up its specific phases, create
    HTTP clients, open CSV writers, and coordinate everything.
    """

    @abstractmethod
    def run(self) -> None:
        """Execute the full scraping pipeline end-to-end."""
        ...


# Goodreads
Now we focus on scraping books and reviews from Goodreads Listopia pages at https://www.goodreads.com/list.

**Pipeline phases:**
- **Phase 1a** (sync): Crawl genre-tagged list index pages to discover Listopia list URLs
- **Phase 1b** (async): Scrape each list page for individual book URLs &rarr; push into a shared queue
- **Phase 2** (async): Scrape each book page for full metadata + up to 30 reviews via `__NEXT_DATA__` JSON

**Rate-limiting quicknotes:**
- ~4-6 concurrent requests with ~1.5s delay is ideal
- 429s come in bursts: the global cool-off in AsyncHttpClient prevents cascading failures
- Cloudflare challenges trigger above ~8 concurrent; we stay well under

## Goodreads Data Models

In [ ]:
@dataclass
class BookListing:
    """Lightweight record from a Goodreads list page.

    Contains just enough info to queue the book for detailed scraping.
    The list-level rating fields let us compare list ratings vs actual book page ratings.
    """
    title: str                   # book title as shown on the list
    url: str                     # full URL to the book page (e.g. https://www.goodreads.com/book/show/12345)
    book_id: str = ""            # numeric Goodreads book ID extracted from the URL
    avg_rating_list: str = ""    # average rating shown on the list (may differ from book page)
    num_ratings_list: str = ""   # number of ratings shown on the list
    list_rank: str = ""          # position on the list (e.g. "1", "2", "3")
    source_list: str = ""        # which list this book came from (for traceability)


@dataclass
class BookDetail:
    """Full metadata scraped from an individual Goodreads book page.

    All fields are strings because some may be empty/missing, and we want
    consistent CSV output without mixed types.
    """
    title: str = ""
    title_complete: str = ""     # includes subtitle if any
    url: str = ""
    book_id: str = ""
    legacy_id: str = ""          # older Goodreads ID format
    description: str = ""        # HTML-stripped book description
    image_url: str = ""          # cover image URL
    author: str = ""
    author_url: str = ""
    avg_rating: str = ""         # from the book page (more accurate than list page)
    ratings_count: str = ""
    reviews_count: str = ""
    rating_dist_1: str = ""      # count of 1-star ratings
    rating_dist_2: str = ""      # count of 2-star ratings
    rating_dist_3: str = ""      # count of 3-star ratings
    rating_dist_4: str = ""      # count of 4-star ratings
    rating_dist_5: str = ""      # count of 5-star ratings
    num_pages: str = ""
    book_format: str = ""        # e.g. "Hardcover", "Paperback", "Kindle Edition"
    publisher: str = ""
    publication_date: str = ""   # YYYY-MM-DD format
    isbn: str = ""
    isbn13: str = ""
    asin: str = ""               # Amazon Standard Identification Number
    language: str = ""
    genres: str = ""             # pipe-separated, e.g. "Fiction|Fantasy|Adventure"
    series_name: str = ""
    series_position: str = ""    # e.g. "1", "2.5"
    content_rating: str = ""
    awards: str = ""             # pipe-separated award names
    source_list: str = ""        # carried over from BookListing
    list_rank: str = ""          # carried over from BookListing


@dataclass
class Review:
    """A single user review scraped from a Goodreads book page.

    Reviews are embedded in the same __NEXT_DATA__ JSON as the book metadata,
    so we get them for free — no extra HTTP request needed.
    """
    book_title: str = ""         # which book this review belongs to
    book_url: str = ""
    reviewer_name: str = ""
    reviewer_id: str = ""
    rating: str = ""             # 1-5 star rating
    review_text: str = ""        # HTML-stripped review body
    created_at: str = ""         # YYYY-MM-DD when review was posted
    updated_at: str = ""         # YYYY-MM-DD when review was last edited
    like_count: str = ""         # how many people found this review helpful
    comment_count: str = ""
    shelves: str = ""            # pipe-separated shelf names (e.g. "read|fiction|favorites")

## Goodreads Configuration
remember every scraper inherits from the BaseScraperConfig class

In [ ]:
# these genre tags drive Phase 1a — we'll crawl /list/tag/{genre} for each one
GENRE_TAGS = [
    "fiction", "non-fiction", "fantasy", "science-fiction", "mystery",
    "romance", "thriller", "horror", "historical-fiction", "biography",
    "history", "classics", "young-adult", "children", "poetry",
    "philosophy", "psychology", "science", "self-help", "business",
    "art", "music", "travel", "crime", "adventure",
    "humor", "memoir", "graphic-novels", "comics", "drama",
]

In [ ]:
@dataclass(frozen=True)
class GoodreadsConfig(BaseScraperConfig):
    """Goodreads-specific settings that extend the shared base config."""
    genre_tags: tuple[str, ...] = tuple(GENRE_TAGS)  # genres to discover lists from
    list_index_pages_per_genre: int = 3   # how many pages of /list/tag/{genre} to crawl (~20 lists per page)
    max_books_per_list: int = 1000          # cap books scraped from any single list
    max_list_pages: int = 5              # how deep to paginate into each list

    target_books: int = 10_000              # stop discovering once we have this many unique books
    max_reviews_per_book: int = 30        # reviews embedded in __NEXT_DATA__ (usually 30 max)

    # separate concurrency for list scraping vs book detail scraping
    list_concurrency: int = 4             # Phase 1b: lighter, fewer concurrent requests
    detail_concurrency: int = 6           # Phase 2: heavier, but 10+ triggers Cloudflare

    books_csv: str = "goodreads-books.csv"
    reviews_csv: str = "goodreads-reviews.csv"
    listings_cache: str = "goodreads-listings-cache.json"  # save discovered books for re-runs

In [ ]:
def build_list_index_urls(
    genres: list[str] | None = None,
    max_pages_per_genre: int = 3,
) -> list[str]:
    """Build URLs for genre-tagged list index pages.

    Example output: https://www.goodreads.com/list/tag/fiction?page=1
    Each page shows ~20 curated lists. We grab a few pages per genre
    to get a diverse set of lists spanning different topics and quality levels.
    """
    base = "https://www.goodreads.com/list/tag"
    tags = genres or GENRE_TAGS
    urls: list[str] = []
    for tag in tags:
        for page in range(1, max_pages_per_genre + 1):
            urls.append(f"{base}/{tag}?page={page}")
    return urls


## Goodreads Parsenrs
Three parsers handle different page types:

1. **ListIndexPageParser** — parses `/list/tag/{genre}` pages to find Listopia list URLs
2. **ListPageParser** — parses `/list/show/{id}` pages to find individual book URLs + basic metadata
3. **BookPageParser** — parses individual book pages by using beautiful soup and extracting the `__NEXT_DATA__` Apollo GraphQL cache
   embedded in a `<script>` tag. This single JSON blob contains the book's full metadata, author info,
   series info, rating distribution, awards, AND up to 30 reviews — all in one request.

In [ ]:
class ListIndexPageParser:
    """Parses a genre-tagged list index page (/list/tag/{genre}).

    These pages contain links like <a class="listTitle" href="/list/show/1.Best_Books_Ever">
    We extract the href to get Listopia list URLs for Phase 1b.
    """

    LIST_URL_RE = re.compile(r"^/list/show/(\d+)")  # matches /list/show/1234

    @staticmethod
    def parse(html: str) -> list[str]:
        """Extract all unique Listopia list URLs from the page."""
        soup = bs4.BeautifulSoup(html, "lxml")  # lxml is 2-5× faster than html.parser
        urls: list[str] = []
        seen: set[str] = set()  # track hrefs to avoid duplicates within the same page
        for a in soup.select("a.listTitle"):  # Goodreads wraps list names in <a class="listTitle">
            href = a.get("href", "")
            m = ListIndexPageParser.LIST_URL_RE.match(href)
            if not m:
                continue  # skip non-list links
            if href in seen:
                continue  # already found this list on this page
            seen.add(href)
            urls.append("https://www.goodreads.com" + href)  # convert relative → absolute URL
        logger.debug("ListIndexPageParser: extracted %d unique list URLs", len(urls))
        return urls

In [ ]:
class ListPageParser:
    """Parses book listings from a single Listopia list page (/list/show/{id}).

    Each book row is a <tr itemtype="http://schema.org/Book"> containing the book
    title, URL, list rank, and a minirating string like "4.35 avg rating — 10,058,107 ratings".
    """

    BOOK_URL_RE = re.compile(r"^/book/show/(\d+)")  # matches /book/show/12345
    # parses "4.35 avg rating — 10,058,107 ratings" into (4.35, 10058107)
    RATING_RE = re.compile(r"([\d.]+)\s*avg rating\s*[—\-]\s*([\d,]+)\s*ratings?")

    @staticmethod
    def parse(html: str, source_list: str = "") -> list[BookListing]:
        """Extract book listings from the list page HTML."""
        soup = bs4.BeautifulSoup(html, "lxml")
        listings: list[BookListing] = []

        # each book is in a schema.org-typed table row
        rows = soup.select('tr[itemtype="http://schema.org/Book"]')
        for row in rows:
            title_link = row.select_one("a.bookTitle")  # <a class="bookTitle" href="/book/show/...">
            if not title_link:
                continue
            href = title_link.get("href", "")
            m = ListPageParser.BOOK_URL_RE.match(href)
            if not m:
                continue
            book_id = m.group(1)  # extract the numeric book ID from the URL

            # title text is inside a nested <span itemprop="name">
            name_el = title_link.select_one('span[itemprop="name"]')
            title = name_el.get_text(strip=True) if name_el else title_link.get_text(strip=True)

            # list rank from <td class="number"> (e.g. "1", "2")
            rank_el = row.select_one("td.number")
            rank = rank_el.get_text(strip=True) if rank_el else ""

            # extract avg rating + count from <span class="minirating">
            avg_rating = ""
            num_ratings = ""
            mini = row.select_one("span.minirating")
            if mini:
                rm = ListPageParser.RATING_RE.search(mini.get_text(" ", strip=True))
                if rm:
                    avg_rating = rm.group(1)                    # e.g. "4.35"
                    num_ratings = rm.group(2).replace(",", "")  # e.g. "10058107" (remove commas)

            listings.append(BookListing(
                title=title,
                url="https://www.goodreads.com" + href,
                book_id=book_id,
                avg_rating_list=avg_rating,
                num_ratings_list=num_ratings,
                list_rank=rank,
                source_list=source_list,
            ))

        return listings

    @staticmethod
    def has_next_page(html: str, current_page: int) -> bool:
        """Check if there's a page N+1 to fetch.

        Goodreads list pagination uses a <div class="pagination"> block
        with numbered <a> elements. If we find a link for current_page+1, there's more.
        """
        soup = bs4.BeautifulSoup(html, "lxml")
        pagination = soup.select_one("div.pagination")
        if not pagination:
            return False  # no pagination element = single-page list
        for a in pagination.select("a"):
            txt = a.get_text(strip=True)
            if txt.isdigit() and int(txt) == current_page + 1:
                return True  # found a link to the next page
        return False

In [ ]:
class BookPageParser:
    """Extracts BookDetail + Reviews from a Goodreads book page.

    The page embeds an Apollo GraphQL cache in <script id="__NEXT_DATA__">
    containing:
      - Book:<id>        → title, description, ISBN, format, genres, series info
      - Work:<id>        → aggregate stats (avg rating, distribution, awards)
      - Contributor:<id> → author name and URL
      - Review:<id>      → up to 30 reviews with text, rating, shelves

    We parse the whole cache once and resolve internal references manually
    (Apollo uses {"__ref": "Key:..."} for cross-references).
    """

    # regex to find the __NEXT_DATA__ JSON blob in the HTML
    NEXT_DATA_RE = re.compile(
        r'<script id="__NEXT_DATA__" type="application/json">\s*'
        r"(\{.*?\})\s*</script>",
        re.DOTALL,  # allow . to match newlines since the JSON spans multiple lines
    )

    @staticmethod
    def parse(html: str, max_reviews: int = 30) -> tuple[dict, list[dict]]:
        """Parse a book page and return (book_detail_dict, list_of_review_dicts)."""
        apollo = BookPageParser._extract_apollo(html)  # get the GraphQL cache dict
        if not apollo:
            logger.warning("No __NEXT_DATA__ payload found")
            return BookPageParser._fallback_from_ld(html), []  # try JSON-LD as fallback

        book = BookPageParser._find_first(apollo, "Book:")   # find the Book object in the cache
        work = BookPageParser._find_first(apollo, "Work:")   # find the Work object (has stats)
        if not book:
            logger.warning("No Book object in Apollo cache, falling back to JSON-LD")
            return BookPageParser._fallback_from_ld(html), []

        detail = BookPageParser._build_detail(book, work, apollo)   # extract metadata
        reviews = BookPageParser._build_reviews(apollo, max_reviews)  # extract reviews
        return detail, reviews

    # ---------- extraction helpers ----------

    @staticmethod
    def _extract_apollo(html: str) -> dict | None:
        """Pull the Apollo GraphQL cache dict from __NEXT_DATA__."""
        m = BookPageParser.NEXT_DATA_RE.search(html)
        if not m:
            return None
        try:
            data = json.loads(m.group(1))  # parse the entire __NEXT_DATA__ JSON
        except json.JSONDecodeError as e:
            logger.warning("Failed to parse __NEXT_DATA__: %s", e)
            return None
        try:
            # the Apollo cache lives at props.pageProps.apolloState
            return data["props"]["pageProps"]["apolloState"]
        except (KeyError, TypeError):
            return None

    @staticmethod
    def _find_first(apollo: dict, prefix: str) -> dict | None:
        """Find the first key in the Apollo cache starting with the given prefix.

        Keys look like 'Book:kca://book/amzn1.gr.book.v1.abc123' — we match by prefix.
        """
        for k, v in apollo.items():
            if k.startswith(prefix):
                return v
        return None

    @staticmethod
    def _resolve(apollo: dict, maybe_ref: dict | None) -> dict | None:
        """Resolve an Apollo cross-reference.

        Apollo uses {"__ref": "Contributor:kca://..."} to link between objects.
        This function follows the reference and returns the actual object.
        """
        if not maybe_ref:
            return None
        if isinstance(maybe_ref, dict) and "__ref" in maybe_ref:
            return apollo.get(maybe_ref["__ref"])  # follow the reference
        return maybe_ref  # already a concrete object, not a reference

    # ---------- book detail ----------

    @staticmethod
    def _build_detail(book: dict, work: dict | None, apollo: dict) -> dict:
        """Build a flat detail dict from the Book + Work Apollo objects."""
        d: dict = {
            "title": book.get("title", "") or "",
            "title_complete": book.get("titleComplete", "") or "",
            "url": book.get("webUrl", "") or "",
            "book_id": book.get("id", "") or "",
            "legacy_id": str(book.get("legacyId", "")),
            "description": BookPageParser._clean_html(book.get("description", "") or ""),
            "image_url": book.get("imageUrl", "") or "",
        }

        # --- author: resolve the primaryContributorEdge reference ---
        pc_edge = book.get("primaryContributorEdge") or {}
        author = BookPageParser._resolve(apollo, pc_edge.get("node"))
        if author:
            d["author"] = author.get("name", "") or ""
            d["author_url"] = author.get("webUrl", "") or ""

        # --- book details: pages, format, publisher, ISBN, etc. ---
        details = book.get("details") or {}
        d["num_pages"] = str(details.get("numPages") or "")
        d["book_format"] = details.get("format", "") or ""
        d["publisher"] = details.get("publisher", "") or ""
        d["publication_date"] = BookPageParser._ms_to_date(details.get("publicationTime"))
        d["isbn"] = details.get("isbn", "") or ""
        d["isbn13"] = details.get("isbn13", "") or ""
        d["asin"] = details.get("asin", "") or ""
        lang = details.get("language") or {}
        d["language"] = lang.get("name", "") if isinstance(lang, dict) else ""

        # --- genres: flatten the bookGenres array into a pipe-separated string ---
        genres = [
            (bg.get("genre") or {}).get("name", "")
            for bg in (book.get("bookGenres") or [])
            if (bg.get("genre") or {}).get("name")
        ]
        d["genres"] = "|".join(genres)  # e.g. "Fiction|Fantasy|Adventure"

        # --- series: resolve the series reference if the book is part of one ---
        series_list = book.get("bookSeries") or []
        if series_list:
            first = series_list[0]
            series_obj = BookPageParser._resolve(apollo, first.get("series"))
            if series_obj:
                d["series_name"] = series_obj.get("title", "") or ""
            d["series_position"] = first.get("userPosition", "") or ""

        # --- work-level stats: ratings, distribution, awards ---
        if work:
            stats = work.get("stats") or {}
            d["avg_rating"] = str(stats.get("averageRating") or "")
            d["ratings_count"] = str(stats.get("ratingsCount") or "")
            d["reviews_count"] = str(stats.get("textReviewsCount") or "")
            dist = stats.get("ratingsCountDist") or []  # array of 5 ints: [1-star, 2-star, ..., 5-star]
            if len(dist) == 5:
                d["rating_dist_1"] = str(dist[0])
                d["rating_dist_2"] = str(dist[1])
                d["rating_dist_3"] = str(dist[2])
                d["rating_dist_4"] = str(dist[3])
                d["rating_dist_5"] = str(dist[4])

            # awards are nested inside work.details.awardsWon
            wdetails = work.get("details") or {}
            awards_won = wdetails.get("awardsWon") or []
            award_strs = []
            for a in awards_won:
                name = a.get("name", "")
                desig = a.get("designation", "")  # e.g. "Winner", "Nominee"
                if name:
                    award_strs.append(f"{name} ({desig})" if desig else name)
            d["awards"] = "|".join(award_strs)

        return d

    @staticmethod
    def _ms_to_date(ms: int | None) -> str:
        """Convert a millisecond epoch timestamp to YYYY-MM-DD string."""
        if not ms:
            return ""
        try:
            ts = int(ms) / 1000  # milliseconds → seconds
            return time.strftime("%Y-%m-%d", time.gmtime(ts))  # UTC to avoid timezone drift
        except (ValueError, OSError, OverflowError):
            return ""  # some pre-1970 dates cause OverflowError on Windows

    @staticmethod
    def _clean_html(s: str) -> str:
        """Strip HTML tags from a string and collapse whitespace."""
        if not s:
            return ""
        text = bs4.BeautifulSoup(s, "lxml").get_text(" ", strip=True)  # strip tags, join with spaces
        return re.sub(r"\s+", " ", text).strip()  # collapse runs of whitespace to single spaces

    # ---------- reviews ----------

    @staticmethod
    def _build_reviews(apollo: dict, max_reviews: int) -> list[dict]:
        """Extract review data from Review:* objects in the Apollo cache."""
        reviews: list[dict] = []
        for k, v in apollo.items():
            if not k.startswith("Review:"):
                continue  # skip non-review objects
            if len(reviews) >= max_reviews:
                break  # respect the cap

            # resolve the reviewer (creator) reference
            creator = BookPageParser._resolve(apollo, v.get("creator"))

            # extract shelf/tag info (what shelves the reviewer put this book on)
            shelving = v.get("shelving") or {}
            taggings = shelving.get("taggings") or []
            shelves = [
                (t.get("tag") or {}).get("name", "")
                for t in taggings
                if (t.get("tag") or {}).get("name")
            ]
            # prepend the base shelf (e.g. "read", "currently-reading")
            base_shelf = (shelving.get("shelf") or {}).get("name", "")
            if base_shelf:
                shelves = [base_shelf] + shelves

            reviews.append({
                "reviewer_name": (creator or {}).get("name", "") or "",
                "reviewer_id": str((creator or {}).get("legacyId", "")),
                "rating": str(v.get("rating") or ""),
                "review_text": BookPageParser._clean_html(v.get("text", "") or ""),
                "created_at": BookPageParser._ms_to_date(v.get("createdAt")),
                "updated_at": BookPageParser._ms_to_date(v.get("updatedAt")),
                "like_count": str(v.get("likeCount") or ""),
                "comment_count": str(v.get("commentCount") or ""),
                "shelves": "|".join(shelves),  # pipe-separated: "read|fiction|favorites"
            })
        return reviews

    # ---------- JSON-LD fallback ----------

    @staticmethod
    def _fallback_from_ld(html: str) -> dict:
        """If __NEXT_DATA__ is missing/broken, extract what we can from JSON-LD.

        JSON-LD is a simpler structured data block (<script type="application/ld+json">)
        that has basic info like title, author, rating, ISBN. It's less complete than
        Apollo but better than nothing.
        """
        d: dict = {}
        soup = bs4.BeautifulSoup(html, "lxml")
        tag = soup.find("script", type="application/ld+json")
        if not tag or not tag.string:
            return d
        try:
            ld = json.loads(tag.string)
        except json.JSONDecodeError:
            return d
        d["title"] = ld.get("name", "") or ""
        d["image_url"] = ld.get("image", "") or ""
        d["book_format"] = ld.get("bookFormat", "") or ""
        d["num_pages"] = str(ld.get("numberOfPages") or "")
        d["language"] = ld.get("inLanguage", "") or ""
        d["isbn"] = ld.get("isbn", "") or ""
        authors = ld.get("author") or []
        if isinstance(authors, list) and authors:
            d["author"] = authors[0].get("name", "") or ""
            d["author_url"] = authors[0].get("url", "") or ""
        agg = ld.get("aggregateRating") or {}
        d["avg_rating"] = str(agg.get("ratingValue") or "")
        d["ratings_count"] = str(agg.get("ratingCount") or "")
        d["reviews_count"] = str(agg.get("reviewCount") or "")
        return d


## Goodreads Discovery
- phase 1a: synchronous - crawl ~90 genre index pages to find listopia list urls
- phase 1b: asynchronous: scrape each listopia list for book rows, pushes BookListing objects into a shared asyncio.Queue as they are found so pahse 2 can start scraping book pages without waiting fo all lists to finish

In [ ]:
# new logger
gr_logger = setup_logger("goodreads-scraper", "goodreads-scraper.log")

In [ ]:
class GoodreadsListDiscoverer(BaseDiscoverer):
    """Phase 1a: discovers Listopia list URLs by crawling genre-tagged index pages."""

    def __init__(self, config: GoodreadsConfig, client: SyncHttpClient | None = None):
        self._config = config
        # use provided client or create a default one with the config's settings
        self._client = client or SyncHttpClient(
            config.user_agents, config.request_delay,
            config.max_retries, config.retry_backoff,
        )
        self._parser = ListIndexPageParser()

    def discover(self) -> list[str]:
        """Crawl genre index pages and return a deduplicated list of Listopia URLs."""
        # build all the genre index page URLs we need to crawl
        index_urls = build_list_index_urls(
            list(self._config.genre_tags),
            self._config.list_index_pages_per_genre,
        )
        gr_logger.info("Phase 1a: crawling %d genre index pages", len(index_urls))

        all_lists: list[str] = []   # ordered list of unique list URLs
        seen: set[str] = set()      # fast lookup for deduplication

        for i, url in enumerate(index_urls, 1):
            try:
                html = self._client.get(url)  # synchronous fetch
            except Exception as e:
                gr_logger.warning("[1a %d/%d] failed %s: %s", i, len(index_urls), url, e)
                continue  # skip failed pages, keep going
            found = self._parser.parse(html)  # extract list URLs from the page
            new = [u for u in found if u not in seen]  # filter out duplicates
            for u in new:
                seen.add(u)
                all_lists.append(u)
            gr_logger.info(
                "[1a %d/%d] %s — +%d lists (total: %d)",
                i, len(index_urls), url.split("/tag/")[-1], len(new), len(all_lists),
            )

        gr_logger.info("Phase 1a complete: %d unique lists discovered", len(all_lists))
        return all_lists

In [ ]:
class GoodreadsListScraper:
    """Phase 1b: scrapes each Listopia list for its book rows.

    Pushes BookListing objects into a shared asyncio.Queue as they're found,
    so Phase 2 can start scraping book detail pages immediately.
    Uses a separate AsyncHttpClient from Phase 2 so list-page traffic
    doesn't steal concurrency from book-page traffic.
    """

    def __init__(self, config: GoodreadsConfig, client: AsyncHttpClient):
        self._config = config
        self._client = client
        self._parser = ListPageParser()

    async def scrape_into_queue(
        self,
        list_urls: list[str],             # Listopia URLs to scrape
        queue: asyncio.Queue,             # where to push discovered BookListings
        seen_book_ids: set[str],          # shared set for cross-list deduplication
    ) -> int:
        """Scrape all lists concurrently and push new BookListings into the queue.

        Returns the total number of NEW (unique) listings pushed.
        Puts a None sentinel into the queue when finished to signal Phase 2.
        """
        gr_logger.info(
            "Phase 1b: scraping %d Listopia lists (target: %d books)",
            len(list_urls), self._config.target_books,
        )

        sem = asyncio.Semaphore(self._config.list_concurrency)  # cap concurrent list scrapes
        total_new = 0        # running count of unique books found across all lists
        lists_done = 0       # how many lists we've finished processing
        lock = asyncio.Lock()  # protects seen_book_ids and total_new from race conditions

        async def process_list(list_url: str) -> int:
            """Scrape one Listopia list, paginating up to max_list_pages."""
            nonlocal total_new, lists_done
            async with sem:  # wait here if we're at max concurrency
                new_here = 0  # books found from THIS list
                source_list = list_url.rstrip("/").split("/")[-1]  # e.g. "1.Best_Books_Ever"

                for page in range(1, self._config.max_list_pages + 1):
                    # check if we've hit the global target
                    async with lock:
                        if len(seen_book_ids) >= self._config.target_books:
                            return new_here  # enough books, stop early

                    page_url = f"{list_url}?page={page}" if page > 1 else list_url
                    try:
                        html = await self._client.get(page_url)
                    except Exception as e:
                        gr_logger.warning("List page failed %s: %s", page_url, e)
                        break  # move on to next list

                    listings = self._parser.parse(html, source_list=source_list)
                    if not listings:
                        break  # empty page = no more books on this list

                    per_list_cap = self._config.max_books_per_list
                    async with lock:  # protect shared state
                        for ls in listings:
                            if new_here >= per_list_cap:
                                break  # hit per-list cap
                            if not ls.book_id or ls.book_id in seen_book_ids:
                                continue  # skip duplicates or invalid IDs
                            seen_book_ids.add(ls.book_id)  # mark as seen
                            total_new += 1
                            new_here += 1
                            queue.put_nowait(ls)  # push into queue for Phase 2
                            if len(seen_book_ids) >= self._config.target_books:
                                break  # global target reached

                    if new_here >= per_list_cap:
                        break  # done with this list

                    # check if there's a next page of this list
                    if not self._parser.has_next_page(html, page):
                        break  # no more pages

                lists_done += 1
                if new_here:
                    gr_logger.info(
                        "[1b %d/%d] %s — +%d books (total queued: %d)",
                        lists_done, len(list_urls), source_list, new_here, total_new,
                    )
                return new_here

        # launch all list-scrape tasks concurrently (semaphore limits actual parallelism)
        tasks = [asyncio.create_task(process_list(u)) for u in list_urls]
        try:
            await asyncio.gather(*tasks, return_exceptions=True)
        finally:
            queue.put_nowait(None)  # sentinel: tells Phase 2 "no more books coming"

        gr_logger.info("Phase 1b complete: %d unique books queued", total_new)
        return total_new

## Goodreads Detail Scraper
Consumes `BookListing` objects from the shared queue, fetches each book page,
parses the `__NEXT_DATA__` JSON for metadata + reviews, and streams results
to CSV files.

In [ ]:
class GoodreadsDetailScraper(BaseDetailScraper):
    """Phase 2: scrapes individual Goodreads book pages for full metadata + reviews."""

    def __init__(self, config: GoodreadsConfig, client: AsyncHttpClient):
        self._config = config
        self._client = client
        self._parser = BookPageParser()

    async def scrape_from_queue(
        self,
        queue: asyncio.Queue,     # BookListing objects arrive here from Phase 1b
        *writers: CsvWriter,      # writers[0] = books CSV, writers[1] = reviews CSV
    ) -> tuple[int, ...]:
        """Consume listings from the queue, scrape each book page, write to CSV.

        Returns (books_scraped, reviews_scraped).
        """
        book_writer, review_writer = writers[0], writers[1]
        flush_every = self._config.flush_every

        # load already-scraped URLs so we can skip them
        already_scraped = book_writer.get_existing_values("url")
        if already_scraped:
            gr_logger.info("Crash-resume: found %d already-scraped books, will skip them", len(already_scraped))

        # buffers — we accumulate rows and flush in batches for efficiency
        book_buf: list[BookDetail] = []
        review_buf: list[Review] = []
        books_done = 0        # successful scrapes
        books_failed = 0      # failed scrapes
        total_reviews = 0     # total reviews collected across all books

        start_time = time.monotonic()  # for calculating scraping speed
        pending: set[asyncio.Task] = set()  # currently in-flight scrape tasks
        list_done = False  # whether Phase 1b has finished (sentinel received)

        while True:
            # --- fill up the pending set with new tasks ---
            while len(pending) < self._config.detail_concurrency and not list_done:
                try:
                    listing = queue.get_nowait()  # non-blocking dequeue
                except asyncio.QueueEmpty:
                    break  # queue is empty right now, wait for more

                if listing is None:
                    list_done = True  # Phase 1b is done, no more books coming
                    gr_logger.info("List-scrape phase complete, draining %d remaining", len(pending))
                    break

                # validate the URL before spending a network request on it
                if not listing.url or not listing.url.startswith("https://www.goodreads.com/book/show/"):
                    gr_logger.debug("Skipping invalid URL: %r for '%s'", listing.url, listing.title)
                    continue

                # skip if already scraped in prev run
                if listing.url in already_scraped:
                    gr_logger.debug("Skipping already-scraped: %s", listing.title)
                    continue


                task = asyncio.create_task(self._scrape_one(listing))  # launch async scrape
                pending.add(task)

            # --- check termination ---
            if not pending and list_done:
                break  # all done!
            if not pending:
                await asyncio.sleep(0.1)  # nothing to do yet, wait briefly
                continue

            # --- wait for at least one task to finish ---
            done, pending = await asyncio.wait(pending, return_when=asyncio.FIRST_COMPLETED)

            for task in done:
                try:
                    detail, reviews, title = task.result()  # unpack successful result
                    book_buf.append(detail)
                    review_buf.extend(reviews)
                    books_done += 1
                    total_reviews += len(reviews)
                    gr_logger.info("[%d done] ✓ %s (%d reviews)", books_done, title, len(reviews))
                except Exception as e:
                    books_failed += 1
                    gr_logger.error("[%d failed] ✗ %s: %s", books_failed, type(e).__name__, e)

            # --- flush buffers to disk periodically ---
            if len(book_buf) >= flush_every:
                book_writer.append(book_buf)
                review_writer.append(review_buf)
                book_buf.clear()
                review_buf.clear()
                elapsed = time.monotonic() - start_time
                rate = books_done / elapsed if elapsed > 0 else 0  # books per second
                gr_logger.info("Flushed to CSV | %d scraped, %d failed (%.2f books/sec)",
                               books_done, books_failed, rate)

        # flush any remaining buffered data
        if book_buf:
            book_writer.append(book_buf)
            review_writer.append(review_buf)

        return books_done, total_reviews

    async def _scrape_one(self, listing: BookListing) -> tuple[BookDetail, list[Review], str]:
        """Scrape a single book's metadata + embedded reviews."""
        html = await self._client.get(listing.url)  # fetch the book page
        detail_dict, review_dicts = self._parser.parse(html, self._config.max_reviews_per_book)

        # merge list-discovery fields into the detail (source_list, list_rank)
        detail_dict.setdefault("url", listing.url)
        detail_dict["source_list"] = listing.source_list
        detail_dict["list_rank"] = listing.list_rank
        if not detail_dict.get("title"):
            detail_dict["title"] = listing.title  # fallback to list title if page title is missing

        # build BookDetail, ignoring any extra keys not in the dataclass
        valid_fields = {f.name for f in fields(BookDetail)}
        filtered = {k: v for k, v in detail_dict.items() if k in valid_fields}
        detail = BookDetail(**filtered)

        # build Review objects, attaching the book title and URL to each one
        book_url = detail.url or listing.url
        book_title = detail.title or listing.title
        reviews = [Review(book_title=book_title, book_url=book_url, **r) for r in review_dicts]

        return detail, reviews, book_title


## Goodreads Pipeline Orchestrator

Wires everything together:
1. Check for cached listings (skip Phases 1a+1b if found)
2. Run Phase 1a (sync discovery) → Phase 1b (async list scraping) → Phase 2 (async detail scraping)
3. Phases 1b and 2 run **concurrently** on the same event loop via a shared queue


In [ ]:
class GoodreadsPipeline(BasePipeline):
    """Top-level orchestrator for the Goodreads scraping pipeline."""

    def __init__(self, config: GoodreadsConfig | None = None):
        self._config = config or GoodreadsConfig()

    def run(self) -> None:
        """Execute the full pipeline. Uses cache if available."""
        cache_path = os.path.join(self._config.output_dir, self._config.listings_cache)
        cached = load_cache(cache_path, BookListing)  # try loading previous discovery results

        if cached:
            gr_logger.info("Loaded %d book listings from cache: %s", len(cached), cache_path)
            gr_logger.info("Skipping Phases 1a+1b, feeding cache directly to scraper")
            asyncio.run(self._run_from_cache(cached))
        else:
            gr_logger.info("No cache found — running full pipeline (Phase 1a → 1b → 2)")
            asyncio.run(self._run_interweaved())

    async def _run_interweaved(self) -> None:
        """Run all phases: discovery feeds a queue that detail scraping consumes."""
        cache_path = os.path.join(self._config.output_dir, self._config.listings_cache)

        # Phase 1a: synchronous discovery of Listopia list URLs
        discoverer = GoodreadsListDiscoverer(self._config)
        list_urls = discoverer.discover()
        if not list_urls:
            gr_logger.error("Phase 1a returned no lists — aborting")
            return

        # shared queue: Phase 1b pushes BookListings, Phase 2 consumes them
        queue: asyncio.Queue[BookListing | None] = asyncio.Queue(maxsize=500)
        seen_book_ids: set[str] = set()  # prevents duplicate books across lists

        # separate HTTP clients for list vs detail scraping (independent rate limiting)
        list_client = AsyncHttpClient(
            self._config.user_agents, self._config.request_delay,
            self._config.max_retries, self._config.retry_backoff,
            self._config.default_rate_limit_wait,
        )
        detail_client = AsyncHttpClient(
            self._config.user_agents, self._config.request_delay,
            self._config.max_retries, self._config.retry_backoff,
            self._config.default_rate_limit_wait,
        )
        await list_client.start(self._config.list_concurrency)
        await detail_client.start(self._config.detail_concurrency)

        # open CSV output files
        out = self._config.output_dir
        book_writer = CsvWriter(os.path.join(out, self._config.books_csv), BookDetail)
        review_writer = CsvWriter(os.path.join(out, self._config.reviews_csv), Review)
        book_writer.open()
        review_writer.open()

        list_scraper = GoodreadsListScraper(self._config, list_client)
        detail_scraper = GoodreadsDetailScraper(self._config, detail_client)

        # tap into the queue to collect listings for caching
        cached_listings: list[BookListing] = []
        original_put = queue.put_nowait
        def tap_put(item):
            if isinstance(item, BookListing):
                cached_listings.append(item)  # save a copy for the cache
            return original_put(item)
        queue.put_nowait = tap_put  # monkey-patch the queue to intercept puts

        try:
            # run Phase 1b (list scraping) and Phase 2 (detail scraping) concurrently
            list_task = asyncio.create_task(
                list_scraper.scrape_into_queue(list_urls, queue, seen_book_ids)
            )
            detail_task = asyncio.create_task(
                detail_scraper.scrape_from_queue(queue, book_writer, review_writer)
            )

            list_total = await list_task         # wait for Phase 1b to finish
            books_n, reviews_n = await detail_task  # wait for Phase 2 to finish

            gr_logger.info(
                "Pipeline done! %d lists → %d books queued → %d scraped, %d reviews",
                len(list_urls), list_total, books_n, reviews_n,
            )
        finally:
            book_writer.close()
            review_writer.close()
            await list_client.close()
            await detail_client.close()
            save_cache(cache_path, cached_listings, dedup_key="book_id")  # persist for re-runs

    async def _run_from_cache(self, listings: list[BookListing]) -> None:
        """Run Phase 2 only, using cached listings from a prior run."""
        # filter out any invalid URLs that might have snuck into the cache
        valid = [ls for ls in listings
                 if ls.url and ls.url.startswith("https://www.goodreads.com/book/show/")]
        if len(valid) < len(listings):
            gr_logger.warning("Filtered %d invalid URLs from cache (%d → %d)",
                              len(listings) - len(valid), len(listings), len(valid))

        # load everything into the queue at once + sentinel
        queue: asyncio.Queue[BookListing | None] = asyncio.Queue()
        for ls in valid:
            queue.put_nowait(ls)
        queue.put_nowait(None)  # sentinel: no more items

        out = self._config.output_dir
        book_writer = CsvWriter(os.path.join(out, self._config.books_csv), BookDetail)
        review_writer = CsvWriter(os.path.join(out, self._config.reviews_csv), Review)
        book_writer.open()
        review_writer.open()

        detail_client = AsyncHttpClient(
            self._config.user_agents, self._config.request_delay,
            self._config.max_retries, self._config.retry_backoff,
            self._config.default_rate_limit_wait,
        )
        await detail_client.start(self._config.detail_concurrency)

        try:
            detail_scraper = GoodreadsDetailScraper(self._config, detail_client)
            books_n, reviews_n = await detail_scraper.scrape_from_queue(
                queue, book_writer, review_writer
            )
            gr_logger.info("Done from cache! %d books, %d reviews", books_n, reviews_n)
        finally:
            book_writer.close()
            review_writer.close()
            await detail_client.close()


# Rotten Tomatoes
Scrapes movies, critic reveiws and audience reviews

**Pipeline phases:**
- **Phase 1** (sync, Playwright): Scroll through JS-rendered browse pages, click "Load More",
  and collect movie URLs. Runs in a background thread because Playwright requires synchronous execution.
- **Phase 2** (async, aiohttp): Scrape each movie page for metadata (JSON-LD + scorecard JSON +
  info section), then fetch critic and audience review pages.

**Key differences from Goodreads:**
- Browse pages are JS-rendered → requires Playwright instead of simple HTTP requests
- Three CSV outputs instead of two (movies, critic reviews, audience reviews)
- Cache uses a dict keyed by lowercase title (for title-based deduplication)


## Rotten Tomatoes Data Models

In [ ]:
@dataclass
class MovieListing:
    """Lightweight record from a browse page tile."""
    title: str                      # movie title from the tile
    url: str                        # full URL (e.g. https://www.rottentomatoes.com/m/the_matrix)
    critic_score_browse: str = ""   # tomatometer shown on the tile (may differ from detail page)
    audience_score_browse: str = "" # audience score shown on the tile


@dataclass
class MovieDetail:
    """Full metadata scraped from an individual movie page.

    Combines data from four sources on the page:
    1. JSON-LD (name, rating, genres, directors, actors)
    2. media-scorecard-json (detailed critic/audience/top-critic scores)
    3. Movie Info section (runtime, box office, distributor)
    4. Consensus divs (critic and audience consensus text)
    """
    title: str = ""
    url: str = ""
    critic_score_browse: str = ""     # from browse tile (for comparison)
    audience_score_browse: str = ""   # from browse tile (for comparison)
    name: str = ""                    # official movie name from JSON-LD
    content_rating: str = ""          # e.g. "PG-13", "R"
    release_date: str = ""
    genres: str = ""                  # pipe-separated: "Action|Adventure|Sci-Fi"
    directors: str = ""               # pipe-separated director names
    actors: str = ""                  # pipe-separated actor names
    tomatometer: str = ""             # critic score (0-100)
    tomatometer_count: str = ""       # number of critic reviews
    critic_avg_rating: str = ""       # average critic rating (e.g. "7.5/10")
    critic_sentiment: str = ""        # "positive", "negative", "mixed"
    critic_certified: str = ""        # whether it's "Certified Fresh"
    critic_liked: str = ""            # count of positive critic reviews
    critic_not_liked: str = ""        # count of negative critic reviews
    top_critic_score: str = ""        # tomatometer from top critics only
    top_critic_count: str = ""
    top_critic_liked: str = ""
    top_critic_not_liked: str = ""
    top_critic_sentiment: str = ""
    top_critic_certified: str = ""
    audience_score: str = ""          # verified audience score (0-100)
    audience_avg_rating: str = ""     # average audience rating (e.g. "4.2/5")
    audience_review_count: str = ""
    audience_sentiment: str = ""
    audience_certified: str = ""
    audience_liked: str = ""
    audience_not_liked: str = ""
    audience_rating_band: str = ""    # distribution across rating bands
    audience_score_type: str = ""     # "verified" or "all"
    audience_all_score: str = ""      # score including unverified reviews
    audience_all_avg_rating: str = ""
    audience_all_review_count: str = ""
    audience_all_liked: str = ""
    audience_all_not_liked: str = ""
    audience_all_sentiment: str = ""
    synopsis: str = ""
    runtime: str = ""                 # e.g. "2h 16m"
    box_office_usa: str = ""
    original_language: str = ""
    release_date_theaters: str = ""
    release_date_streaming: str = ""
    distributor: str = ""
    production_co: str = ""
    critics_consensus: str = ""       # one-line critic summary
    audience_consensus: str = ""      # one-line audience summary


@dataclass
class CriticReview:
    """A single critic review from a movie's /reviews page."""
    movie_title: str = ""
    movie_url: str = ""
    critic_name: str = ""
    publication: str = ""    # e.g. "The New York Times"
    sentiment: str = ""      # "positive" or "negative"
    rating: str = ""         # original rating text (e.g. "B+", "3.5/4")
    review_text: str = ""


@dataclass
class AudienceReview:
    """A single audience review from a movie's /reviews?type=user page."""
    movie_title: str = ""
    movie_url: str = ""
    user_name: str = ""
    score: str = ""          # star rating (0.5 to 5.0)
    review_text: str = ""
    verified: str = ""       # "true" if the reviewer has a verified ticket purchase


## Rotten Tomatoes Configuration & URL Generation

In [ ]:
def _build_category_urls() -> tuple[str, ...]:
    """Generate a comprehensive set of RT browse page URLs.

    We combine multiple filter axes to get diverse movies:
      Tier 0: Unfiltered sort-only pages (broadest mix, catches worst-rated films)
      Tier 1: Rating filter × sort order (rating diversity)
      Tier 2: Genre × sort order (fills genre gaps)
      Tier 3: Genre × rating combos (catches niche intersections)

    This produces ~700+ unique URLs. We process them in tier order, and the
    adaptive skipping logic in the browse collector will bail on low-yield pages.
    """
    base = "https://www.rottentomatoes.com/browse"
    paths = ["movies_at_home", "movies_in_theaters"]  # two main movie sections

    rating_filters = [
        "audience:verified_hot", "audience:upright", "audience:spilled",  # audience ratings
        "critics:certified_fresh", "critics:fresh", "critics:rotten",      # critic ratings
    ]
    genre_filters = [
        "action", "adventure", "animation", "comedy", "crime",
        "documentary", "drama", "fantasy", "history", "horror",
        "kids_and_family", "music", "mystery_and_thriller", "romance",
        "sci_fi", "sports", "war", "western", "biography", "anime",
    ]
    sort_orders = [
        "popular", "newest",
        "critic_highest", "critic_lowest",
        "audience_highest", "audience_lowest",
    ]
    urls: list[str] = []

    # tier 0: no filters, just sort order — surfaces the broadest mix
    for path in paths:
        for sort in sort_orders:
            urls.append(f"{base}/{path}/sort:{sort}")

    # tier 1: rating filter × sort — ensures we get spilled AND fresh movies
    for path in paths:
        for rating in rating_filters:
            for sort in sort_orders:
                urls.append(f"{base}/{path}/{rating}~sort:{sort}")

    # tier 2: genre × sort — fills genre diversity gaps
    for path in paths:
        for genre in genre_filters:
            for sort in sort_orders:
                urls.append(f"{base}/{path}/genres:{genre}~sort:{sort}")

    # tier 3: genre × rating combos (popular sort only to keep count manageable)
    for path in paths:
        for genre in genre_filters:
            for rating in rating_filters:
                urls.append(f"{base}/{path}/{rating}~genres:{genre}~sort:popular")

    return tuple(urls)


@dataclass(frozen=True)
class RottenTomatoesConfig(BaseScraperConfig):
    """RT-specific settings extending the shared base config."""
    category_urls: tuple[str, ...] = _build_category_urls()  # all browse page URLs

    target_movies: int = 10_000              # stop browsing once we have this many unique movies
    max_scroll_per_category: int = 8       # max "Load More" clicks per browse page
    scroll_pause: float = 0.3              # settle time after each "Load More" click

    max_critic_reviews: int = 50           # critic reviews to scrape per movie
    max_audience_reviews: int = 50         # audience reviews to scrape per movie

    detail_concurrency: int = 10           # Phase 2 concurrency (10+ triggers RT blocking)
    min_new_ratio: float = 0.05            # skip browse page if < 5% of movies are new

    movies_csv: str = "rotten-tomatoes-movies.csv"
    critic_reviews_csv: str = "rotten-tomatoes-critic-reviews.csv"
    audience_reviews_csv: str = "rotten-tomatoes-audience-reviews.csv"
    listings_cache: str = "movie-listings-cache.json"

    # RT uses a single user agent (Playwright browser context matches it)
    user_agent: str = (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/146.0.0.0 Safari/537.36"
    )


## Rotten Tomatoes Parsers

Three parser classes handle different RT page types:

1. **BrowsePageParser** — extracts movie tiles from browse/discovery pages
2. **MoviePageParser** — extracts metadata from four sources on a movie page (JSON-LD, scorecard JSON, info section, consensus divs)
3. **ReviewPageParser** — extracts critic and audience reviews from review pages

In [ ]:
class BrowsePageParser:
    """Parses movie tiles from a RT browse/discovery page."""

    @staticmethod
    def parse(html: str) -> list[MovieListing]:
        soup = bs4.BeautifulSoup(html, "lxml")
        movies: list[MovieListing] = []

        # each movie tile is a <div data-qa="discovery-media-list-item">
        tiles = soup.select('[data-qa="discovery-media-list-item"]')
        for i, tile in enumerate(tiles):
            # try the caption link first (most common)
            link = tile.select_one('[data-qa="discovery-media-list-item-caption"]')
            href = link.get("href", "") if link else ""

            # fallback: some tiles have the link on a parent <a>
            if not href:
                parent_link = tile.find_parent("a")
                if parent_link:
                    href = parent_link.get("href", "")

            # fallback: look for any <a> with an /m/ href inside the tile
            if not href:
                any_movie_link = tile.select_one('a[href^="/m/"]')
                if any_movie_link:
                    href = any_movie_link.get("href", "")

            # extract the movie title
            title_el = (link or tile).select_one('[data-qa="discovery-media-list-item-title"]')
            if not title_el:
                title_el = tile.select_one('[data-qa="discovery-media-list-item-title"]')
            title = title_el.get_text(strip=True) if title_el else ""

            # only keep movie links (href starts with /m/)
            if not href or not href.startswith("/m/"):
                continue

            # extract scores from the tile
            critic_el = (link or tile).select_one('[slot="criticsScore"]')
            audience_el = (link or tile).select_one('[slot="audienceScore"]')

            movies.append(MovieListing(
                title=title,
                url="https://www.rottentomatoes.com" + href,
                critic_score_browse=critic_el.get_text(strip=True) if critic_el else "",
                audience_score_browse=audience_el.get_text(strip=True) if audience_el else "",
            ))

        return movies


class MoviePageParser:
    """Extracts structured metadata from an individual RT movie page.

    Combines four data sources into a single flat dict:
    1. JSON-LD: basic movie info (name, rating, genres, cast)
    2. media-scorecard-json: detailed scores (tomatometer, audience, top critics)
    3. Movie info section: runtime, box office, distributor, etc.
    4. Consensus divs: critic and audience consensus text
    """

    @staticmethod
    def parse(html: str) -> dict:
        soup = bs4.BeautifulSoup(html, "lxml")
        details: dict = {}
        MoviePageParser._parse_json_ld(soup, details)           # source 1
        MoviePageParser._parse_scorecard_json(soup, details)    # source 2
        MoviePageParser._parse_movie_info_section(soup, details)  # source 3
        MoviePageParser._parse_consensus(soup, details)         # source 4
        return details

    @staticmethod
    def _parse_json_ld(soup: bs4.BeautifulSoup, details: dict) -> None:
        """Extract basic info from the JSON-LD structured data block."""
        ld_tag = soup.find("script", type="application/ld+json")
        if not ld_tag or not ld_tag.string:
            return
        try:
            ld = json.loads(ld_tag.string)
        except (json.JSONDecodeError, TypeError):
            return

        details["name"] = ld.get("name", "")
        details["content_rating"] = ld.get("contentRating", "")
        details["release_date"] = ld.get("dateCreated", "")

        # genres can be a list or a single string
        genres = ld.get("genre", [])
        details["genres"] = "|".join(genres) if isinstance(genres, list) else str(genres)

        # directors can be a list of dicts or a single dict
        directors = ld.get("director", [])
        if isinstance(directors, list):
            details["directors"] = "|".join(d.get("name", "") for d in directors)
        elif isinstance(directors, dict):
            details["directors"] = directors.get("name", "")

        # actors is always a list of dicts
        actors = ld.get("actor", [])
        if isinstance(actors, list):
            details["actors"] = "|".join(a.get("name", "") for a in actors)

        # aggregate rating (fallback — scorecard JSON is more detailed)
        agg = ld.get("aggregateRating", {})
        if not details.get("tomatometer"):
            details["tomatometer"] = str(agg.get("ratingValue", ""))
        if not details.get("tomatometer_count"):
            details["tomatometer_count"] = str(agg.get("ratingCount", ""))

    @staticmethod
    def _parse_scorecard_json(soup: bs4.BeautifulSoup, details: dict) -> None:
        """Extract detailed scoring data from the media-scorecard-json script tag.

        This is the richest data source — it has critic, audience, and top-critic
        scores along with liked/not-liked counts and sentiment.
        """
        tag = soup.find("script", id="media-scorecard-json")
        if not tag or not tag.string:
            return
        try:
            data = json.loads(tag.string.strip())
        except (json.JSONDecodeError, TypeError):
            return

        # --- critic scores ---
        critics = data.get("criticsScore", {})
        details["tomatometer"] = str(critics.get("score", ""))
        details["tomatometer_count"] = str(critics.get("ratingCount", critics.get("reviewCount", "")))
        details["critic_avg_rating"] = str(critics.get("averageRating", ""))
        details["critic_sentiment"] = critics.get("sentiment", "")
        details["critic_certified"] = str(critics.get("certified", ""))
        details["critic_liked"] = str(critics.get("likedCount", ""))
        details["critic_not_liked"] = str(critics.get("notLikedCount", ""))

        # --- top critic scores (from overlay section) ---
        overlay = data.get("overlay", {})
        top = overlay.get("criticsTop", {})
        details["top_critic_score"] = str(top.get("score", ""))
        details["top_critic_count"] = str(top.get("ratingCount", top.get("reviewCount", "")))
        details["top_critic_liked"] = str(top.get("likedCount", ""))
        details["top_critic_not_liked"] = str(top.get("notLikedCount", ""))
        details["top_critic_sentiment"] = top.get("sentiment", "")
        details["top_critic_certified"] = str(top.get("certified", ""))

        # --- audience scores ---
        audience = data.get("audienceScore", {})
        details["audience_score"] = str(audience.get("score", ""))
        details["audience_avg_rating"] = str(audience.get("averageRating", ""))
        details["audience_review_count"] = str(audience.get("reviewCount", ""))
        details["audience_sentiment"] = audience.get("sentiment", "")
        details["audience_certified"] = str(audience.get("certified", ""))
        details["audience_liked"] = str(audience.get("likedCount", ""))
        details["audience_not_liked"] = str(audience.get("notLikedCount", ""))
        details["audience_rating_band"] = audience.get("bandedRatingCount", "")
        details["audience_score_type"] = audience.get("scoreType", "")

        # --- audience "all" (including unverified reviews) ---
        aud_all = overlay.get("audienceAll", {})
        details["audience_all_score"] = str(aud_all.get("score", ""))
        details["audience_all_avg_rating"] = str(aud_all.get("averageRating", ""))
        details["audience_all_review_count"] = str(aud_all.get("reviewCount", ""))
        details["audience_all_liked"] = str(aud_all.get("likedCount", ""))
        details["audience_all_not_liked"] = str(aud_all.get("notLikedCount", ""))
        details["audience_all_sentiment"] = aud_all.get("sentiment", "")

        if not details.get("synopsis"):
            details["synopsis"] = data.get("description", "")

    @staticmethod
    def _parse_movie_info_section(soup: bs4.BeautifulSoup, details: dict) -> None:
        """Extract runtime, box office, distributor, etc. from the info section."""
        section = soup.select_one('[data-qa="section:media-info"]')
        if not section:
            return

        # synopsis (fallback if not found in scorecard JSON)
        synopsis_el = section.select_one('[data-qa="synopsis-value"]')
        if synopsis_el and not details.get("synopsis"):
            details["synopsis"] = synopsis_el.get_text(strip=True)

        # map human-readable labels to our field names
        label_map = {
            "runtime": "runtime",
            "box office (gross usa)": "box_office_usa",
            "original language": "original_language",
            "release date (theaters)": "release_date_theaters",
            "release date (streaming)": "release_date_streaming",
            "distributor": "distributor",
            "production co": "production_co",
        }
        for item in section.select('[data-qa="item"]'):
            label_el = item.select_one('[data-qa="item-label"]')
            if not label_el:
                continue
            field_name = label_map.get(label_el.get_text(strip=True).lower())
            if not field_name:
                continue  # label doesn't match any field we care about
            value_els = item.select('[data-qa="item-value"]')
            details[field_name] = ", ".join(v.get_text(strip=True) for v in value_els)

    @staticmethod
    def _parse_consensus(soup: bs4.BeautifulSoup, details: dict) -> None:
        """Extract the one-line critic and audience consensus text."""
        for div_id, key in [
            ("#critics-consensus", "critics_consensus"),
            ("#audience-consensus", "audience_consensus"),
        ]:
            div = soup.select_one(div_id)
            if div:
                p = div.find("p")  # consensus text is in a <p> inside the div
                if p:
                    details[key] = p.get_text(strip=True)


class ReviewPageParser:
    """Parses critic and audience reviews from RT review pages."""

    @staticmethod
    def parse_critic_reviews(html: str, max_reviews: int = 10) -> list[CriticReview]:
        """Extract critic reviews from /reviews page. Reviews are in <review-card> elements."""
        soup = bs4.BeautifulSoup(html, "lxml")
        reviews: list[CriticReview] = []
        for card in soup.select("review-card"):
            if len(reviews) >= max_reviews:
                break
            name_el = card.select_one('[slot="name"]')       # critic's name
            pub_el = card.select_one('[slot="publication"]')  # publication name
            review_el = card.select_one('[slot="review"]')    # review text
            sentiment_el = card.select_one("score-icon-critics")  # fresh/rotten icon
            rating_text = ""
            rating_el = card.select_one('[slot="rating"]')
            if rating_el:
                for span in rating_el.select("span"):
                    t = span.get_text(strip=True)
                    if t:
                        rating_text = t  # e.g. "B+", "3.5/4", "8/10"
                        break
            reviews.append(CriticReview(
                critic_name=name_el.get_text(strip=True) if name_el else "",
                publication=pub_el.get_text(strip=True) if pub_el else "",
                sentiment=sentiment_el.get("sentiment", "") if sentiment_el else "",
                rating=rating_text,
                review_text=review_el.get_text(strip=True) if review_el else "",
            ))
        return reviews

    @staticmethod
    def parse_audience_reviews(html: str, max_reviews: int = 10) -> list[AudienceReview]:
        """Extract audience reviews from /reviews?type=user page."""
        soup = bs4.BeautifulSoup(html, "lxml")
        reviews: list[AudienceReview] = []
        for card in soup.select("review-card"):
            if len(reviews) >= max_reviews:
                break
            name_el = card.select_one('[slot="name"]')
            rating_el = card.select_one("rating-stars-group")  # star rating component
            content_el = card.select_one('[slot="content"]')   # review text
            reviews.append(AudienceReview(
                user_name=name_el.get_text(strip=True) if name_el else "",
                score=rating_el.get("score", "") if rating_el else "",  # e.g. "4.5"
                review_text=content_el.get_text(strip=True) if content_el else "",
                verified=card.get("verified-review", "false"),  # "true"/"false"
            ))
        return reviews


## RT Browse Collector (Phase 1 — Playwright)

The browse pages on Rotten Tomatoes are JS-rendered — you can't just `requests.get()`
them because the movie tiles are loaded dynamically. We use **Playwright** to:

1. Navigate to each browse page
2. Dismiss cookie banners and overlay modals
3. Click "Load More" to reveal more movies
4. Wait adaptively for new tiles to appear (instead of fixed sleeps)
5. Skip low-yield pages (if < 5% of movies are new after the first scroll)

This runs in a **background thread** because Playwright requires synchronous execution,
while the rest of the pipeline runs on an async event loop. The two are bridged via
`loop.call_soon_threadsafe(queue.put_nowait, listing)`.


In [ ]:
rt_logger = setup_logger("rt-scraper", "rt-scraper.log")


class RTBrowseCollector(BaseDiscoverer):
    """Scrolls through RT browse pages using Playwright to collect movie URLs."""

    def __init__(self, config: RottenTomatoesConfig):
        self._config = config
        self._parser = BrowsePageParser()

    def discover(self) -> list[MovieListing]:
        """Standalone discovery (not interweaved) — returns a flat list."""
        seen: dict[str, MovieListing] = {}
        self._browse_all(seen)
        return list(seen.values())

    def collect_into_queue(
        self,
        queue: asyncio.Queue,                    # where to push discovered MovieListings
        loop: asyncio.AbstractEventLoop,         # the async event loop (for thread-safe puts)
    ) -> dict[str, MovieListing]:
        """Crawl browse pages and feed new listings into the async queue.

        Runs on a background thread. Uses loop.call_soon_threadsafe() to
        safely push items from this thread into the asyncio queue.
        Returns the full seen dict for caching.
        """
        seen: dict[str, MovieListing] = {}  # keyed by lowercase title for dedup
        total_urls = len(self._config.category_urls)

        with sync_playwright() as pw:
            browser = pw.chromium.launch(headless=True, timeout=30000)  # headless Chrome
            context = browser.new_context(user_agent=self._config.user_agent)

            rt_logger.info("%d category URLs to crawl (target: %d movies)",
                           total_urls, self._config.target_movies)

            for idx, cat_url in enumerate(self._config.category_urls, 1):
                if len(seen) >= self._config.target_movies:
                    rt_logger.info("Reached target of %d unique movies", self._config.target_movies)
                    break

                label = cat_url.split("/browse/")[-1]  # short label for logging
                rt_logger.info("[browse %d/%d] %s", idx, total_urls, label)

                try:
                    new_listings = self._scroll_and_collect(context.new_page(), cat_url, seen)
                    # push each new listing into the async queue (thread-safe)
                    for listing in new_listings:
                        loop.call_soon_threadsafe(queue.put_nowait, listing)
                    if new_listings:
                        rt_logger.info("  +%d new (total: %d)", len(new_listings), len(seen))
                except Exception as e:
                    rt_logger.warning("Browse error on %s: %s", label, e)

                # save incremental cache every 50 URLs in case of crash
                if idx % 50 == 0:
                    self._save_incremental_cache(seen)

            browser.close()

        # signal "no more movies" to Phase 2
        loop.call_soon_threadsafe(queue.put_nowait, None)
        self._save_incremental_cache(seen)  # final cache save
        rt_logger.info("Browse complete: %d unique movies collected", len(seen))
        return seen

    def _browse_all(self, seen: dict[str, MovieListing]) -> None:
        """Browse all category pages without a queue (for standalone mode)."""
        with sync_playwright() as pw:
            browser = pw.chromium.launch(headless=True)
            context = browser.new_context(user_agent=self._config.user_agent)
            for idx, cat_url in enumerate(self._config.category_urls, 1):
                if len(seen) >= self._config.target_movies:
                    break
                try:
                    self._scroll_and_collect(context.new_page(), cat_url, seen)
                except Exception as e:
                    rt_logger.warning("Browse error: %s", e)
            browser.close()

    def _save_incremental_cache(self, seen: dict[str, MovieListing]) -> None:
        """Save current listings to a partial cache file during collection."""
        cache_path = os.path.join(self._config.output_dir, self._config.listings_cache)
        try:
            os.makedirs(os.path.dirname(cache_path) or ".", exist_ok=True)
            data = {key: asdict(val) for key, val in seen.items()}
            with open(cache_path, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False)
        except Exception as e:
            rt_logger.warning("Failed to save incremental cache: %s", e)

    def _dismiss_overlays(self, page: Page) -> None:
        """Dismiss cookie consent banners and app modals that block clicks.

        RT uses OneTrust for cookie consent and custom overlay-base modals.
        If we don't dismiss these, the "Load More" button click fails.
        """
        # try OneTrust cookie banner buttons
        for selector in (
            "#onetrust-accept-btn-handler",
            "#onetrust-reject-all-handler",
            "#onetrust-close-btn-container button",
        ):
            try:
                btn = page.query_selector(selector)
                if btn and btn.is_visible():
                    btn.click()
                    page.wait_for_timeout(500)
                    return
            except Exception:
                pass

        # try custom RT overlay modals
        for selector in (
            'overlay-base[data-rtappmanager="overlayBase:close"]',
            "overlay-base",
        ):
            try:
                overlay = page.query_selector(selector)
                if overlay and overlay.is_visible():
                    close_btn = page.query_selector(f'{selector} [aria-label="Close"]')
                    if close_btn and close_btn.is_visible():
                        close_btn.click()
                        page.wait_for_timeout(500)
                        return
            except Exception:
                pass

        # last resort: press Escape to dismiss any overlay
        try:
            page.keyboard.press("Escape")
            page.wait_for_timeout(300)
        except Exception:
            pass

    def _scroll_and_collect(
        self, page: Page, url: str, seen: dict[str, MovieListing],
    ) -> list[MovieListing]:
        """Scroll a single browse page, clicking "Load More" to reveal movies.

        Returns a list of NEW (previously unseen) MovieListings.
        Also adds them to the `seen` dict for cross-page deduplication.
        """
        page.goto(url, wait_until="domcontentloaded")  # navigate to the page
        page.wait_for_timeout(3000)  # wait for initial JS rendering
        self._dismiss_overlays(page)  # clear any blocking overlays

        new_listings: list[MovieListing] = []
        for scroll_num in range(self._config.max_scroll_per_category):
            if len(seen) >= self._config.target_movies:
                break  # reached global target

            html = page.content()  # get the current DOM state
            listings = self._parser.parse(html)  # extract movie tiles

            new_this_scroll = 0
            for m in listings:
                key = m.title.lower().strip()  # normalize for dedup
                if key and key not in seen:
                    seen[key] = m
                    new_listings.append(m)
                    new_this_scroll += 1

            # adaptive skipping: if the first scroll yields < 5% new, bail
            # (this page is mostly duplicates of movies we already have)
            if scroll_num == 0 and len(listings) > 10:
                ratio = new_this_scroll / len(listings) if listings else 0
                if ratio < self._config.min_new_ratio:
                    rt_logger.debug(
                        "Skipping low-yield page (%d/%d new): %s",
                        new_this_scroll, len(listings), url,
                    )
                    break

            # find and click the "Load More" button
            btn = page.query_selector('[data-qa="dlp-load-more-button"]')
            if not btn or not btn.is_visible():
                break  # no more content to load

            # count tiles before click so we can wait for new ones to appear
            tiles_before = len(page.query_selector_all('[data-qa="discovery-media-list-item"]'))

            try:
                self._dismiss_overlays(page)  # clear overlays before clicking
                btn.scroll_into_view_if_needed()
                btn.click(timeout=10000)
            except Exception:
                # fallback: use JavaScript to click (works even if the button is obscured)
                try:
                    page.evaluate(
                        'document.querySelector(\'[data-qa="dlp-load-more-button"]\')?.click()'
                    )
                except Exception:
                    break

            # adaptive wait: wait for new tiles to appear instead of a fixed sleep
            try:
                page.wait_for_function(
                    f'document.querySelectorAll(\'[data-qa="discovery-media-list-item"]\').length > {tiles_before}',
                    timeout=5000,
                )
            except Exception:
                pass  # timeout is fine — maybe no more content

            page.wait_for_timeout(int(self._config.scroll_pause * 1000))  # short settle time

        page.close()  # close the tab to free resources
        return new_listings


## Rotten Tomatoes Detail Scraper

In [ ]:
class RTDetailScraper(BaseDetailScraper):
    """Scrapes individual movie pages + review pages using asyncio + aiohttp."""

    def __init__(self, config: RottenTomatoesConfig, client: AsyncHttpClient):
        self._config = config
        self._client = client
        self._movie_parser = MoviePageParser()
        self._review_parser = ReviewPageParser()

    async def scrape_from_queue(
        self,
        queue: asyncio.Queue,     # MovieListing objects arrive here from the browse collector
        *writers: CsvWriter,      # [0]=movies, [1]=critic reviews, [2]=audience reviews
    ) -> tuple[int, ...]:
        """Consume movie listings, scrape each, write to three CSV files."""
        movie_writer, critic_writer, audience_writer = writers[0], writers[1], writers[2]
        flush_every = self._config.flush_every

        already_scraped = movie_writer.get_existing_values("url")
        if already_scraped:
            rt_logger.info("Crash-resume: found %d already-scraped movies, will skip them", len(already_scraped))

        movie_buf: list[MovieDetail] = []
        critic_buf: list[CriticReview] = []
        audience_buf: list[AudienceReview] = []
        movies_done = 0
        movies_failed = 0
        total_critics = 0
        total_audience = 0

        start_time = time.monotonic()
        pending: set[asyncio.Task] = set()  # in-flight scrape tasks
        browse_done = False  # set to True when we receive the None sentinel

        while True:
            # --- fill pending tasks up to the concurrency limit ---
            while len(pending) < self._config.detail_concurrency and not browse_done:
                try:
                    listing = queue.get_nowait()
                except asyncio.QueueEmpty:
                    break

                if listing is None:
                    browse_done = True  # browse collector is done
                    rt_logger.info("Browse phase complete, draining %d remaining", len(pending))
                    break

                if not listing.url or not listing.url.startswith("https://www.rottentomatoes.com/m/"):
                    continue  # skip non-movie URLs

                if listing.url in already_scraped:
                    rt_logger.debug("Skipping already-scraped: %s", listing.title)
                    continue

                task = asyncio.create_task(self._scrape_one(listing))
                pending.add(task)

            if not pending and browse_done:
                break  # all work done
            if not pending:
                await asyncio.sleep(0.1)
                continue

            # --- wait for at least one task to finish ---
            done, pending = await asyncio.wait(pending, return_when=asyncio.FIRST_COMPLETED)

            for task in done:
                try:
                    detail, critics, audience, title = task.result()
                    movie_buf.append(detail)
                    critic_buf.extend(critics)
                    audience_buf.extend(audience)
                    movies_done += 1
                    total_critics += len(critics)
                    total_audience += len(audience)
                    rt_logger.info("[%d done] ✓ %s", movies_done, title)
                except Exception as e:
                    movies_failed += 1
                    rt_logger.error("[%d failed] ✗ %s: %s", movies_failed, type(e).__name__, e)

            # --- flush to CSV periodically ---
            if len(movie_buf) >= flush_every:
                movie_writer.append(movie_buf)
                critic_writer.append(critic_buf)
                audience_writer.append(audience_buf)
                movie_buf.clear()
                critic_buf.clear()
                audience_buf.clear()
                elapsed = time.monotonic() - start_time
                rate = movies_done / elapsed if elapsed > 0 else 0
                rt_logger.info("Flushed to CSV | %d scraped, %d failed (%.1f/sec)",
                               movies_done, movies_failed, rate)

        # flush remaining
        if movie_buf:
            movie_writer.append(movie_buf)
            critic_writer.append(critic_buf)
            audience_writer.append(audience_buf)

        return movies_done, total_critics, total_audience

    async def _scrape_one(
        self, listing: MovieListing
    ) -> tuple[MovieDetail, list[CriticReview], list[AudienceReview], str]:
        """Scrape one movie: metadata page + critic reviews + audience reviews."""
        url = listing.url

        # 1. movie metadata (JSON-LD + scorecard + info section + consensus)
        html = await self._client.get(url)
        parsed = self._movie_parser.parse(html)
        detail = MovieDetail(
            title=listing.title, url=url,
            critic_score_browse=listing.critic_score_browse,
            audience_score_browse=listing.audience_score_browse,
            **parsed,
        )

        # 2. critic reviews (separate page: /m/{slug}/reviews)
        critic_reviews: list[CriticReview] = []
        try:
            reviews_html = await self._client.get(url.rstrip("/") + "/reviews")
            critic_reviews = self._review_parser.parse_critic_reviews(
                reviews_html, self._config.max_critic_reviews
            )
            for r in critic_reviews:
                r.movie_title = listing.title  # attach movie context to each review
                r.movie_url = url
        except Exception:
            pass  # review page failed — not critical, we still have the metadata

        # 3. audience reviews (separate page: /m/{slug}/reviews?type=user)
        audience_reviews: list[AudienceReview] = []
        try:
            audience_html = await self._client.get(url.rstrip("/") + "/reviews?type=user")
            audience_reviews = self._review_parser.parse_audience_reviews(
                audience_html, self._config.max_audience_reviews
            )
            for r in audience_reviews:
                r.movie_title = listing.title
                r.movie_url = url
        except Exception:
            pass

        return detail, critic_reviews, audience_reviews, listing.title


## Rotten Tomatoes Pipeline Orchestrator
Wires everything together:
1. Check for cached listings (skip browse phase if found)
2. Run Playwright browse collector in a **background thread**
3. Run async detail scraper on the **main event loop**
4. Bridge them via a shared asyncio.Queue

In [ ]:
class RottenTomatoesPipeline(BasePipeline):
    """Top-level orchestrator for the Rotten Tomatoes scraping pipeline."""

    def __init__(self, config: RottenTomatoesConfig | None = None):
        self._config = config or RottenTomatoesConfig()

    def run(self) -> None:
        cache_path = os.path.join(self._config.output_dir, self._config.listings_cache)
        cached = self._load_cache(cache_path)

        if cached:
            rt_logger.info("Loaded %d movie URLs from cache: %s", len(cached), cache_path)
            asyncio.run(self._run_from_cache(cached))
        else:
            rt_logger.info("No cache found — running interweaved browse + scrape")
            asyncio.run(self._run_interweaved())

    async def _run_interweaved(self) -> None:
        """Run browse (thread) + detail scraping (async) simultaneously."""
        queue: asyncio.Queue[MovieListing | None] = asyncio.Queue()
        loop = asyncio.get_running_loop()

        # open all three CSV writers
        out = self._config.output_dir
        movie_writer = CsvWriter(os.path.join(out, self._config.movies_csv), MovieDetail)
        critic_writer = CsvWriter(os.path.join(out, self._config.critic_reviews_csv), CriticReview)
        audience_writer = CsvWriter(os.path.join(out, self._config.audience_reviews_csv), AudienceReview)
        movie_writer.open()
        critic_writer.open()
        audience_writer.open()

        # create the async HTTP client for Phase 2
        detail_client = AsyncHttpClient(
            self._config.user_agents, self._config.request_delay,
            self._config.max_retries, self._config.retry_backoff,
            self._config.default_rate_limit_wait,
        )
        await detail_client.start(self._config.detail_concurrency)
        detail_scraper = RTDetailScraper(self._config, detail_client)

        # run the Playwright browse collector in a background thread
        collector = RTBrowseCollector(self._config)
        import concurrent.futures
        executor = concurrent.futures.ThreadPoolExecutor(max_workers=1)
        browse_future = loop.run_in_executor(
            executor, collector.collect_into_queue, queue, loop
        )

        try:
            # consume from queue and scrape details concurrently
            movies_n, critics_n, audience_n = await detail_scraper.scrape_from_queue(
                queue, movie_writer, critic_writer, audience_writer
            )
        finally:
            movie_writer.close()
            critic_writer.close()
            audience_writer.close()
            await detail_client.close()

        await browse_future  # wait for browse thread to finish
        executor.shutdown(wait=False)
        rt_logger.info("Done! %d movies, %d critic reviews, %d audience reviews",
                       movies_n, critics_n, audience_n)

    async def _run_from_cache(self, listings: dict[str, MovieListing]) -> None:
        """Run Phase 2 only, using cached listings."""
        valid = {k: v for k, v in listings.items()
                 if v.url and v.url.startswith("https://www.rottentomatoes.com/m/")}
        if len(valid) < len(listings):
            rt_logger.warning("Filtered %d invalid URLs from cache (%d → %d)",
                              len(listings) - len(valid), len(listings), len(valid))

        queue: asyncio.Queue[MovieListing | None] = asyncio.Queue()
        for listing in valid.values():
            queue.put_nowait(listing)
        queue.put_nowait(None)  # sentinel

        out = self._config.output_dir
        movie_writer = CsvWriter(os.path.join(out, self._config.movies_csv), MovieDetail)
        critic_writer = CsvWriter(os.path.join(out, self._config.critic_reviews_csv), CriticReview)
        audience_writer = CsvWriter(os.path.join(out, self._config.audience_reviews_csv), AudienceReview)
        movie_writer.open()
        critic_writer.open()
        audience_writer.open()

        detail_client = AsyncHttpClient(
            self._config.user_agents, self._config.request_delay,
            self._config.max_retries, self._config.retry_backoff,
            self._config.default_rate_limit_wait,
        )
        await detail_client.start(self._config.detail_concurrency)

        try:
            detail_scraper = RTDetailScraper(self._config, detail_client)
            movies_n, critics_n, audience_n = await detail_scraper.scrape_from_queue(
                queue, movie_writer, critic_writer, audience_writer
            )
        finally:
            movie_writer.close()
            critic_writer.close()
            audience_writer.close()
            await detail_client.close()

        rt_logger.info("Done! %d movies, %d critic reviews, %d audience reviews",
                       movies_n, critics_n, audience_n)

    @staticmethod
    def _load_cache(path: str) -> dict[str, MovieListing] | None:
        """Load cached movie listings from JSON. Returns None if no cache."""
        if not os.path.exists(path):
            return None
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            return {key: MovieListing(**val) for key, val in data.items()}
        except (json.JSONDecodeError, TypeError, KeyError) as e:
            rt_logger.warning("Cache file corrupt, ignoring: %s", e)
            return None


# Pipeline Registry & Execution

The registry maps short names to pipeline factory functions. Adding a new
source is simple, just add one entry here,  the factory function handles the import
so dependencies are only loaded when actually needed.

In [ ]:
PIPELINES: dict[str, callable] = {
    "goodreads":       lambda: GoodreadsPipeline(),
    "rottentomatoes":  lambda: RottenTomatoesPipeline(),
    # --- add new sources here ---
    # "example-amazon":        lambda: example-AmazonPipeline(),
}

In [ ]:
def run_pipelines(names: list[str] | None = None) -> None:
    """Run one or more scraping pipelines by name.

    If names is None, runs ALL registered pipelines.
    """
    requested = names or list(PIPELINES.keys())

    unknown = [n for n in requested if n not in PIPELINES]
    if unknown:
        print(f"Unknown source(s): {', '.join(unknown)}")
        print(f"Available: {', '.join(PIPELINES.keys())}")
        return

    for name in requested:
        logger.info("=" * 60)
        logger.info("Starting pipeline: %s", name)
        logger.info("=" * 60)
        try:
            pipeline = PIPELINES[name]()  # create the pipeline instance
            pipeline.run()                # execute end-to-end
        except Exception as e:
            logger.error("Pipeline '%s' failed: %s: %s", name, type(e).__name__, e)
            raise

## Run a Pipeline

Uncomment the line for the scraper you want to run. Each pipeline is
independent — you can run them in any order or combination.


In [ ]:
# --- Uncomment ONE of the following to run ---

# Run only the Goodreads scraper:
# run_pipelines(["goodreads"])

# Run only the Rotten Tomatoes scraper:
# run_pipelines(["rottentomatoes"])


# Run sample unknown:
# run_pipelines(["amazon"])

# Run both scrapers:
# run_pipelines(["goodreads", "rottentomatoes"])

# Run ALL registered scrapers:
# run_pipelines()
